# Isochrone Catchment Pipeline - Bayern

1. Fix params, change `SPEC` accordingly
2. Data loading: `panel`, admin. shapefile, and pop. raster
3. Optional: Restrict to a Bundesland; consult `config.py` for a list of Schlussel indicating Bundeslaende. 
4. Snap stores (given specification) to the road network (server must be running at the given `OSRM` address). Results cached; same Shop Type, year specification and road network can be re-run by pointing to the resulting cache file. Not terribly expensive.
5. Reduce, addapt and snap pop cells: If the specification-corresponding parquet does not exist, load pop raster, addapt resolution (lower resolution where catchments are wide, given population, store locations), and snap to the road network. This is computationally taxing; the pop. raster is very fine definition, and millions of cells need to snap to the network. They cannot be passed in bulk, like the distance-to querries, and therefore we querry the server millions of times. This is why the local implementation is of most value, because this would take hours if not days if we had to wait a sec after every querry. Snapping does not move the cell, but masks it with the road-approximate cell; then it inherits its attributes when the nearest store is found. After snapping, duplicate masks (cell masks that coincide) are dropped (aggregated, and the corresponding cells inherit the survivors' attributes)
6. Test server for the big task 
7. Travel times: 


Order matters in two places:

1. Stores are snapped before `adapt_resolution`, because resolution is chosen
   from the distance to the nearest store and that should be the routable position.
2. `m1` / `m2` are saved the moment they exist. They are produced by
   `group_by` + `with_row_index`, and Polars does not guarantee group order across
   runs -- regenerating them later yields *different* node ids than the ones baked
   into `cand` / `dur` / the assignments, and the mismatch fails silently.

Only `snap` and `travel_times` touch OSRM; both cache, so a re-run is cheap.

**Bump `SPEC` whenever you change a reduction or candidate parameter.** 
`c2`, `mapping` and `matrix.npz` are one artefact split across three files; a stale
matrix beside regenerated cells is the main way this pipeline produces wrong
answers, and only some of those combinations raise on their own.

## 0. Config

In [ ]:

import sys, time
from pathlib import Path
import numpy as np
import polars as pl
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))
from scripts.tdlinx.data_loading import load_geodata

import scripts.isochrones.network_voronoi_2 as nv
import scripts.isochrones.market_metrics as M

# ---- paths -----------------------------------------------------------------
SHARE   = Path(r"X:\Project-Shares\germart-lite\germart-papaio")
PANEL   = SHARE / "panel"
GHSL    = r"X:\Project-Shares\germart-lite\germart-papaio\edeka_project_2\data\ghsl_raster_files\ghs_pop_germany_clipped.tif"
WORK    = SHARE / r"edeka_project_2\data\catchments"
OUT     = SHARE / r"edeka_project_2\outputs\catchments"
OSRM        = "http://localhost:5000"      # or http://192.168.70.123:5000
# SPEC namespaces every derived artefact. Change any reduction/candidate parameter
# below and you MUST bump this: cells, mapping and matrix are one object split
# across three files, and mixing versions is the main way this pipeline goes wrong.

# ---- parameters ------------------------------------------------------------
YEARS       = [2024]                       # add more; only assign() re-runs per year
SHOP_TYPES  = ["Supermarkt", "Discount Geschäft"]
K           = 20                           # 20 certifies rank 1; use 60 if you need ranks 2-3
RANKS       = 3                            # 3 => Huff choice set; 1 => headline catchment only
BETA        = 2.0                          # Huff distance decay -- an assumption, not an estimate
ATTRACT     = "VKFL"                       # Huff attractiveness; set None for time-only
LAND        = "Bayern"
KREIS_ID    = "AGS5"                       # matches MARKET_DEFINING_COLS and the panel
KREIS_NAME  = "GEN"

#------------------------#
SPEC    = "bayern_supermarkets_2024"
#------------------------#

CACHE   = OUT / SPEC / "cache"
CKPT    = OUT / SPEC / "ckpt"
SPECDIR = WORK / SPEC; OUT = OUT / SPEC; FIGURES = OUT / "figures"
for d in (WORK, CACHE, SPECDIR, CKPT, OUT, FIGURES): d.mkdir(parents=True, exist_ok=True)



## 1. Load panel + geodata

In [ ]:

panel = pl.read_parquet(PANEL)
gdf, kreis_gdf, bundesland_gdf = load_geodata()

panel = panel.filter(pl.col("Shop Type").is_in(SHOP_TYPES), pl.col("year").is_in(YEARS))
print(f"panel: {panel.height:,} rows | {panel['TDLinx'].n_unique():,} stores | "
      f"years {panel['year'].min()}-{panel['year'].max()}")

## 1b. Restrict to Bayern

Clips the inputs: a Bayern-only raster, Bayern Kreise, and a panel
filtered to Bayern stores. Everything downstream runs unchanged.

`GHSL` and `kreis_gdf` are rebound here, so `load_cells`, `rasterize_regions` and
`write_label_raster` all see the clipped grid and stay mutually consistent.

In [ ]:

import geopandas as gpd, rasterio
from rasterio.mask import mask as rio_mask

AGS5_PREFIX = "09"                        # Bayern; see SCHLUESSEL_TO_ABR in config

# ---- Kreis layer keyed on AGS5, dissolved from the Gemeinde layer ----------
kreis_gdf = (gdf.assign(AGS5=gdf["ARS"].str[:5])
                .dissolve(by="AGS5", aggfunc="first")
                .reset_index())
if KREIS_NAME not in kreis_gdf.columns:    # fall back to the key if no name column
    kreis_gdf[KREIS_NAME] = kreis_gdf["AGS5"]

bayern_kreis = kreis_gdf[kreis_gdf["AGS5"].str.startswith(AGS5_PREFIX)]
assert len(bayern_kreis), f"no AGS5 starting {AGS5_PREFIX!r}"
bayern = bayern_kreis.dissolve().geometry.iloc[0]

# ---- raster -> Bayern extent ----------------------------------------------
GHSL_BY = SPECDIR / f"ghs_pop_{LAND.lower()}_100m.tif"
if not GHSL_BY.exists():
    with rasterio.open(GHSL) as src:
        geom = gpd.GeoSeries([bayern], crs=kreis_gdf.crs).to_crs(src.crs).geometry.values
        img, tf = rio_mask(src, geom, crop=True, filled=True, nodata=0)
        prof = src.profile
    prof.update(height=img.shape[1], width=img.shape[2], transform=tf,
                nodata=0, compress="lzw")
    with rasterio.open(GHSL_BY, "w", **prof) as dst:
        dst.write(img[0], 1)
    print(f"wrote {GHSL_BY.name}  {prof['height']}x{prof['width']}")

GHSL      = GHSL_BY                        # everything downstream uses this grid
kreis_gdf = bayern_kreis

# ---- panel -> Bayern -------------------------------------------------------
panel = panel.filter(pl.col("Bundesland") == "Bayern")

print(f"panel : {panel.height:,} rows | {panel['TDLinx'].n_unique():,} stores")
print(f"Kreise: {len(kreis_gdf)}")
print(f"cross-check vs Bundesland column: "
      f"{sorted(panel['Bundesland'].unique().to_list())}")

## 2. Stores -> snapped

`drop_over_m=300`: a real operating retail location has a road within 300 m. Anything further is a geocode to check, and because a store is a candidate for thousands of cells one bad row corrupts a whole region.

In [ ]:
stores = nv.prepare_stores(panel, keep=("Gruppe", "Shop Type"))
stores = nv.snap(stores, base_url=OSRM, cache_path=CACHE, drop_over_m=300)
stores.write_parquet(SPECDIR / f"stores_snapped_{SPEC}.parquet")

In [ ]:
# stores = pl.read_parquet(SPECDIR / f"stores_snapped_{SPEC}.parquet")

## 3. Cells -> reduced + snapped

Run once. Every later run reads the parquet and never touches OSRM.

In [ ]:

cells_path, map_path = SPECDIR / f"cells_solve_{SPEC}.parquet", SPECDIR / f"mapping_{SPEC}.parquet"

if cells_path.exists() and map_path.exists():
    cells_orig = nv.load_cells(GHSL)                 # local, deterministic, no OSRM
    c2      = pl.read_parquet(cells_path)
    mapping = pl.read_parquet(map_path)
    print(f"loaded {c2.height:,} solve nodes from disk")
else:
    cells_orig = nv.load_cells(GHSL)
    c1, m1 = nv.adapt_resolution(cells_orig, stores)
    c1     = nv.snap(c1, base_url=OSRM, cache_path=CACHE)
    c2, m2 = nv.dedupe(c1, tol_m=0.0)                # 0 = exact, no approximation
    mapping = nv.compose(m1, m2)

    # save immediately: group_by order is not stable across runs
    m1.write_parquet(SPECDIR / f"map_adapt_{SPEC}.parquet")
    m2.write_parquet(SPECDIR / f"map_dedupe_{SPEC}.parquet")
    mapping.write_parquet(map_path)
    c2.write_parquet(cells_path)

print(f"{cells_orig.height:,} raster cells -> {c2.height:,} solve nodes "
      f"({c2.height/cells_orig.height:.1%})")

## 4. Preflight

10000 is a good value for `max_table_size` in the server. If observed speed exceeds `V_MAX_KMH`, the certification bound is unsound -- raise it in the module.

In [ ]:
nv.preflight(c2, base_url=OSRM)

## 5. Travel times

Checkpointed: kill and re-run and it resumes. `progress_every` prints a measured ETA after the first batches.

In [ ]:

matrix_path = SPECDIR / f"matrix_{SPEC}.npz"

if matrix_path.exists():
    z    = nv.load_matrix(matrix_path, stores=stores, cells=c2)
    cand, dur = z["candidates"], z["durations"]
    print(f"loaded matrix {dur.shape}")
else:
    cand   = nv.candidates(c2, stores, k=K)
    dur, _ = nv.travel_times(c2, stores, cand, base_url=OSRM,
                             checkpoint=CKPT / f"{SPEC}", max_workers=8, progress_every=50)
    nv.save_matrix(matrix_path, c2, stores, cand, dur)

nv.certification_report(cand, dur, ranks=RANKS)

In [ ]:

matrix_path = SPECDIR / f"matrix_{SPEC}_temp.npz"

if matrix_path.exists():
    z    = nv.load_matrix(matrix_path, stores=stores, cells=c2)
    cand, dur = z["candidates"], z["durations"]
    print(f"loaded matrix {dur.shape}")
else:
    cand_temp   = nv.candidates(c2, stores, k=K)
    dur_temp, _ = nv.travel_times(c2, stores, cand, base_url=OSRM,
                             checkpoint=CKPT / f"{SPEC}", max_workers=8, progress_every=50)
    nv.save_matrix(matrix_path, c2, stores, cand_temp, dur_temp)

nv.certification_report(cand_temp, dur_temp, ranks=RANKS)

If rank-1 certification is below ~99%, widen and re-solve only the
uncertified nodes:

```python
cand, dur, ok = nv.refine(c2, stores, cand, dur, k_new=60, rank=1,
                          base_url=OSRM, checkpoint=CKPT / "refine")
nv.save_matrix(matrix_path, c2, stores, cand, dur, cert=ok)
```

In [ ]:
# cand_ref_ref, dur_ref_ref, ok_ref_ref = nv.refine(c2, stores, cand, dur, k_new=80, rank=1,
#                           base_url=OSRM, checkpoint=CKPT / "refine_refine")
# nv.save_matrix(matrix_path, c2, stores, cand_ref_ref, dur_ref_ref, cert=ok_ref_ref)
# nv.certification_report(cand_ref_ref, dur_ref_ref, ranks=RANKS)

## 6. Assign per year + save raster/parquet

`write_label_raster` needs `cells_orig` (it carries row/col) and `mapping`, so the tif comes out at full 100 m resolution regardless of how far the solve was coarsened. The GeoTIFF stores integer codes; the TDLinx lookup is a parquet sidecar.

In [ ]:

assignments = {}
for year in YEARS:
      a = nv.assign(c2, stores, cand, dur, year=year, ranks=RANKS)
      assignments[year] = a

      alive = ((stores["first_year"].to_numpy() <= year)
                  & (stores["last_year"].to_numpy() >= year))
      n_orph = nv.orphans(cand, dur, alive).size
      print(f"\n=== {year} ===  {a.filter(pl.col('rank')==1).height:,} nodes assigned, "
            f"{int(alive.sum()):,} stores alive, {n_orph:,} orphan node(s)")
      print(nv.rank_coverage(a))

      for name, obj in (
      ("catchments", nv.catchments(a)),
      ("assignments", a),
      ):
            out_dir = OUT / name
            out_dir.mkdir(parents=True, exist_ok=True)
            obj.write_parquet(out_dir / f"{name}_{SPEC}_{year}.parquet")

            OUT.mkdir(parents=True, exist_ok=True)
            nv.write_label_raster(
            cells_orig,
            a,
            mapping,
            GHSL,
            OUT / f"nearest_store_{SPEC}_{year}.tif",
            )

## 7. Metrics

In [ ]:
labels, lookup = M.rasterize_regions(kreis_gdf, GHSL, cells_orig,
                                     id_col=KREIS_ID, name_col=KREIS_NAME)

metrics = {}
for year in YEARS:
    a  = assignments[year]
    st = stores
    if ATTRACT and ATTRACT in panel.columns:
        st = stores.join(nv.store_year_attrs(panel, year, [ATTRACT]),
                         on="store_id", how="left")
    attract = ATTRACT if (ATTRACT and ATTRACT in st.columns) else None

    alloc  = M.allocate(a, st, rule="huff", attract_col=attract, beta=BETA)
    ca     = M.expand_to_cells(alloc, cells_orig, mapping, labels=labels)
    shares = M.market_shares(ca, st, by="Gruppe")
    conc   = M.hhi(shares)
    local  = M.local_hhi(a, st, attract_col=attract, beta=BETA)
    totals = M.store_totals(ca, st, by="Gruppe")
    metrics[year] = dict(stores=st, alloc=ca, shares=shares, hhi=conc,
                         local=local, totals=totals)

    for name, df in (("shares", shares), ("hhi", conc),
                     ("local_hhi", local), ("gruppe_totals", totals)):
        out_dir = OUT / name
        out_dir.mkdir(parents=True, exist_ok=True)
        df.write_parquet(out_dir / f"{name}_{SPEC}_{year}.parquet")

## 8. Printed summary

In [ ]:

for year in YEARS:
    m = metrics[year]
    conc, tot, loc = m["hhi"], m["totals"], m["local"]
    named = conc.join(lookup, on="region_code", how="left").filter(
        pl.col("region_code") > 0)

    print(f"\n{'='*68}\n{year}\n{'='*68}")
    print(f"population allocated : {m['alloc']['pop_alloc'].sum():,.0f}")
    print(f"Kreise               : {named.height}")
    print(f"HHI (Gruppe)         : median {named['hhi'].median():,.0f}  "
          f"p10 {named['hhi'].quantile(0.1):,.0f}  p90 {named['hhi'].quantile(0.9):,.0f}")
    print(f"CR3                  : median {named['cr3'].median():.1%}")
    n_hi = named.filter(pl.col("hhi") > 2000).height
    print(f"Kreise above HHI 2000: {n_hi} ({n_hi/named.height:.0%})")
    print(f"local HHI (catchment): median {loc['local_hhi'].median():,.0f}, "
          f"mean rivals {loc['mean_rivals'].mean():.2f}")

    print("\nNational Gruppe shares")
    print(tot.select("Gruppe", pl.col("share").round(4), "pop").head(10))

    print("\nMost concentrated Kreise")
    print(named.select(KREIS_NAME, "hhi", "cr1", "cr3", "n_players", "pop")
               .sort("hhi", descending=True).head(10))

## 9. Maps

In [ ]:

year = YEARS[-1]
m = metrics[year]

fig, axes = plt.subplots(1, 3, figsize=(21, 9))
for ax, (col, cmap, ttl) in zip(axes, [
        ("hhi", "OrRd",  f"HHI (Gruppe), {year}"),
        ("cr1", "PuBu",  f"CR1 -- largest chain's share, {year}"),
        ("n_players", "YlGn", f"Distinct Gruppen present, {year}")]):
    M.plot_choropleth(m["hhi"], kreis_gdf, lookup, id_col=KREIS_ID,
                      value_col=col, cmap=cmap, title=ttl, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
# National (or widest-possible) catchment map, coloured by Gruppe
M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                  boundaries=kreis_gdf, title=f"Nearest-store catchments, {year}", save_path=OUT / "figures" / f"catchments_{SPEC}_{year}.png")
plt.show()

In [ ]:
# Zoom to individual Kreise
for name in kreis_gdf["GEN"]:
# for name in ["Regensburg"]:
    M.plot_catchments(assignments[year], c2, m["stores"], by="Gruppe",
                      region=kreis_gdf, region_col=KREIS_NAME, region_name=name,
                      title=f"{name}, {year}", save_path=OUT / "figures" / f"catchments_{SPEC}_{name}_{year}.png",)
    plt.show()


## 10. Robustness: does the finding survive a different demand model?

Winner-take-all vs Huff, and Huff across a range of decay elasticities. `beta` is an assumption.

In [ ]:

st = metrics[year]["stores"]
attract = ATTRACT if (ATTRACT and ATTRACT in st.columns) else None
a = assignments[year]

rows = []
specs = [("winner-take-all", dict(rule="wta"))] + \
        [(f"huff beta={b}", dict(rule="huff", attract_col=attract, beta=b))
         for b in (1.5, 2.0, 2.5)]

for label, kw in specs:
    ca = M.expand_to_cells(M.allocate(a, st, **kw), cells_orig, mapping,
                           labels=labels)
    h  = M.hhi(M.market_shares(ca, st, by="Gruppe")).filter(pl.col("region_code") > 0)
    t  = M.store_totals(ca, st, by="Gruppe")
    rows.append({"spec": label,
                 "hhi_median": h["hhi"].median(),
                 "cr3_median": h["cr3"].median(),
                 "share_top1": t["share"][0],
                 "top1": t["Gruppe"][0]})

print(pl.DataFrame(rows))